<div style="border-top: 1px solid rgba(128, 128, 128, 0.45); padding-top: 10px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Discussion &middot; about 5 min</span></div>

## <span style="font-family: Newsreader, Georgia, serif; font-size: 34px; font-weight: 400;">6. Discussion</span>

<div style="font-size: 19px; line-height: 1.5; max-width: 44em;">The explanations are exact about how the classifier head combined its concepts, and unchecked about whether those concepts are what the image shows. This section shows that gap on our six examples, traces it to the training setup and to the labels the method depends on, and lists what would close it.</div>

Two terms used below:

<div><div style="display: inline-block; vertical-align: top; width: 47%; margin-right: 4%; border-top: 3px solid #4978a7; padding-top: 8px;"><div style="font-family: Newsreader, Georgia, serif; font-size: 22px; margin-bottom: 4px;">Scored patch</div><div style="font-size: 15px; line-height: 1.5;">A patch of the 14 &times; 14 grid that contains a CUB part keypoint. These are the only patches the explanation loss scores. CUB marks 15 parts, so at most 15 of the 196 patches in an image are scored; in our six images, 7 to 13 are.</div></div><div style="display: inline-block; vertical-align: top; width: 47%; border-top: 3px solid #4978a7; padding-top: 8px;"><div style="font-family: Newsreader, Georgia, serif; font-size: 22px; margin-bottom: 4px;">Concentration penalty</div><div style="font-size: 15px; line-height: 1.5;">The training term <code>attention_sparsity</code> (weight 0.5) that rewards each patch for putting its attention on few concepts. It applies to all 196 patches, scored or not. The paper's loss has no such term.</div></div></div>

### 6.1 Faithful is not the same as correct

Section 5 showed that the concept bars rebuild every class score exactly. That guarantee covers two things: which concept slots the head attended to, and how much each slot counted toward each class. It does not cover a third: whether what a slot responds to matches its name. The name "leg color: black" is a training target. The explanation loss applies it at the patches where CUB marks a leg, and nowhere else. On every other patch the slot can respond to whatever helps classification, and the explanation still reports it under that name.

The six examples show this happening:

- **Yellow-headed Blackbird (image 622, correct, 94.2%).** The largest single contribution is "leg color: black", +3.15 of the 4.01 logit margin. The image has no scored leg patch, and the map in Figure 5.1 is bright on the background and dark on the bird. The prediction is right; the reason reported is a leg the model is not looking at.
- **Horned Grebe called Eared Grebe (image 2904, wrong, 95.9%).** The largest push toward the wrong species is "upperparts color: grey", +4.45, attended across the water as well as the body.
- **Attention is not support either.** Of the 30 most-attended global concepts in the six examples (five per image), 12 push toward the other class. On the albatross, "medium size" draws the third-most attention and votes against it.

So a reader of these explanations learns exactly what the head did with each named slot. Whether the slot saw what its name says has to be checked separately.

### 6.2 Why?

1. **Only scored patches are supervised.** A patch gets a concept target only if a CUB part keypoint falls in it; `spatial_concepts_cost` drops every other patch. At most 15 of 196 patches per image are scored.
2. **Every patch votes.** The head averages the class scores of all 196 patches (`out_s.mean(1)`), so an unscored patch shapes the prediction as much as a scored one.
3. **Every patch is pushed to commit.** The concentration penalty rewards each patch for putting its attention on few concepts. In our examples the median patch puts half its attention on a single concept.

So about 185 patches per image must pick a concept with no prior knowledge, and whatever they pick counts just as much as a scored patch.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; instant: cached tensors, no model &middot; change <code>SHOW_TOP</code> and re-run</span></div>

In [ ]:
# Figure 6.1: how few patches the explanation loss scores, and where the unscored patches put their attention.
SHOW_TOP = 3  # concepts named in the printed summary

explanations = json.loads((ARTIFACT_DIR / "examples.json").read_text())
with np.load(ARTIFACT_DIR / "selected_examples.npz") as saved:
    spatial_attention, spatial_targets = saved["spatial_attention"], saved["spatial_targets"]
examples = explanations["examples"]
n_images, n_patches, n_spatial = spatial_attention.shape
N_PARTS = 15  # CUB marks 15 part locations per image; each falls in one patch


def concept_label(attribute):
    """'has_leg_color::black' -> 'leg color: black'; size ranges are dropped."""
    family, value = attribute.replace("has_", "").split("::")
    return "{}: {}".format(family.replace("_", " "), value.split("_(")[0].replace("_", " "))


spatial_names = [concept_label(name) for name in explanations["spatial_attributes"]]
np.testing.assert_allclose(spatial_attention.sum(axis=2), 1.0, atol=1e-4)  # each patch's weights sum to 1

rows, pooled = [], np.zeros(n_spatial)
for i, sample in enumerate(examples):
    scored = ~np.isnan(spatial_targets[i]).all(axis=1)  # patches holding a part keypoint: the only ones the loss sees
    assert 0 < scored.sum() <= N_PARTS
    unscored_mean = spatial_attention[i, ~scored].mean(axis=0)  # a mean of rows that sum to 1, so it sums to 1
    scored_mean = spatial_attention[i, scored].mean(axis=0)
    pooled += spatial_attention[i, ~scored].sum(axis=0)
    rows.append({
        "image_id": sample["image_id"], "correct": sample["correct"], "scored": int(scored.sum()),
        "unscored_top": int(unscored_mean.argmax()), "unscored_share": float(unscored_mean.max()),
        "scored_top": int(scored_mean.argmax()), "scored_share": float(scored_mean.max()),
        "unscored_ranked": [int(j) for j in np.argsort(-unscored_mean)[:SHOW_TOP]], "unscored_mean": unscored_mean,
    })
pooled /= pooled.sum()
even_share = 1 / n_spatial
median_peak = float(np.median(spatial_attention.max(axis=2)))  # the largest weight in each patch's row

fig, (ax_count, ax_share) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.7]},
                                         constrained_layout=True)
y = np.arange(n_images)
ax_count.barh(y, [n_patches] * n_images, color="0.88")
ax_count.barh(y, [r["scored"] for r in rows], color=PALETTE["neg"])
for row, r in zip(y, rows):
    ax_count.text(r["scored"] + 3, row, "{} of {}".format(r["scored"], n_patches), va="center", fontsize=9)
ax_count.set_yticks(y)
ax_count.set_yticklabels(["image {} · {}".format(r["image_id"], "correct" if r["correct"] else "wrong") for r in rows])
ax_count.invert_yaxis()
ax_count.set(xlim=(0, n_patches), xlabel="Patches in the 14 × 14 grid", title="Patches the explanation loss scores")
ax_count.grid(False, axis="y")

height = 0.38
ax_share.barh(y - height / 2, [r["unscored_share"] for r in rows], height=height, color=PALETTE["accent"],
              label="unscored patches, no part keypoint")
ax_share.barh(y + height / 2, [r["scored_share"] for r in rows], height=height, color=PALETTE["neg"],
              label="scored patches")
for row, r in zip(y, rows):
    ax_share.text(r["unscored_share"] + 0.01, row - height / 2, spatial_names[r["unscored_top"]], va="center", fontsize=9)
    ax_share.text(r["scored_share"] + 0.01, row + height / 2, spatial_names[r["scored_top"]], va="center", fontsize=9,
                  alpha=0.7)
ax_share.axvline(even_share, color="0.3", linestyle=":", linewidth=1)
ax_share.text(even_share + 0.008, n_images - 0.5, "even split, 1/{}".format(n_spatial), fontsize=8.5, color="0.3",
              va="center")
ax_share.set_yticks(y)
ax_share.set_yticklabels([])
ax_share.invert_yaxis()
ax_share.set(xlim=(0, 1), xlabel="Share of the group's attention taken by its most-used concept",
             title="Where each group of patches puts its attention")
ax_share.legend(loc="lower right", fontsize=9)
ax_share.grid(False, axis="y")
fig.suptitle("Unscored patches outnumber scored ones 14 to 1, and one concept takes most of their attention", fontsize=13)
plt.show()

unscored_range = (min(r["unscored_share"] for r in rows), max(r["unscored_share"] for r in rows))
scored_range = (min(r["scored_share"] for r in rows), max(r["scored_share"] for r in rows))
leg_black = sum(spatial_names[r["unscored_top"]] == "leg color: black" for r in rows)
caption("<b>Figure 6.1</b> Left: patches that carry a CUB part keypoint, the only ones the explanation loss scores. "
        "Right: for each image, the share of attention taken by the single most-used concept, averaged over the "
        "unscored patches (terracotta) and over the scored patches (blue-grey), with that concept named. Scored patches "
        "follow targets that differ by part, so their average is spread ({:.0%} to {:.0%}). Unscored patches have no "
        "target, and in every image a single concept takes {:.0%} to {:.0%} of their attention: <i>leg color: black</i> in "
        "{} of {}. Cached tensors from the disjoint run's selected checkpoint.".format(
            *scored_range, *unscored_range, leg_black, n_images))
order = np.argsort(-pooled)[:SHOW_TOP]
print("Unscored patches, pooled over the six images, give their attention to: "
      + "; ".join("{} {:.0%}".format(spatial_names[j], pooled[j]) for j in order)
      + " (an even split would be {:.1%} each).".format(even_share))
print("The median patch puts {:.0%} of its attention on one concept.".format(median_peak))
for r in rows:
    print("  image {:>4}: unscored patches' top concepts: ".format(r["image_id"])
          + ", ".join("{} {:.0%}".format(spatial_names[j], r["unscored_mean"][j]) for j in r["unscored_ranked"]))

**"leg color: black".** Over all 196 patches of the six images, "leg color: black" takes a third of the spatial attention and is the largest weight in half of the patches; it is the top concept over the whole image in four of the six.

But it is not a frequent label: a quarter of CUB images have it, rank 33 of the 95 spatial concepts, while "eye color: black" is in 84% of images and does not spread. 

The difference is likely caused by what the slot is trained on. A leg is thin, so the 16-pixel patch at a leg keypoint is mostly ground, grass or pavement with a line of leg across it. The slot named "leg color: black" learns that appearance, and every unscored patch that looks like ground matches it: its attention rises with distance from the bird. The two water images use other concepts the same way: the sea around the albatross goes to "upperparts color: black", the lake around the grebe to "upperparts color: grey".

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; instant: cached tensors and images, no model &middot; change <code>CONCEPT</code> to follow another slot</span></div>

In [ ]:
# Figure 6.2: what the "leg color: black" slot is trained on, and how its attention grows with distance from the bird.
import pandas as pd

CONCEPT = "has_leg_color::black"
CONTRAST = "has_eye_color::black"  # the most frequent spatial label, for comparison
PATCH, N_THUMBS = 16, 2  # 16-px patches; leg patches shown per image

with np.load(ARTIFACT_DIR / "selected_examples.npz") as saved:
    images, spatial_attention, spatial_targets = saved["images"], saved["spatial_attention"], saved["spatial_targets"]
explanations = json.loads((ARTIFACT_DIR / "examples.json").read_text())
examples, spatial_attributes = explanations["examples"], explanations["spatial_attributes"]
n_images, n_patches, n_spatial = spatial_attention.shape
grid = int(round(n_patches ** 0.5))
c = spatial_attributes.index(CONCEPT)
family = CONCEPT.split("::")[0]
family_idx = [k for k, name in enumerate(spatial_attributes) if name.startswith(family)]

# How often the label occurs in CUB, for every spatial concept.
attribute_ids = pd.read_csv(DATA_DIR / "attributes.txt", sep=" ", header=None, names=["attr", "name"]).set_index("name")["attr"]
label_rate = pd.read_csv(DATA_DIR / "CUB_200_2011" / "attributes" / "image_attribute_labels.txt", sep=r"\s+", header=None,
                         engine="c", names=["img", "attr", "present", "certainty", "time"], on_bad_lines="skip"
                         ).groupby("attr")["present"].mean()
frequency = pd.Series({name: label_rate[attribute_ids[name]] for name in spatial_attributes}).sort_values(ascending=False)

# Attention on CONCEPT by Chebyshev distance from the nearest patch that holds a keypoint.
yy, xx = np.divmod(np.arange(n_patches), grid)
distances, keypoint_patches, leg_patches = [], [], []
for i in range(n_images):
    scored = ~np.isnan(spatial_targets[i]).all(axis=1)
    ys, xs = np.divmod(np.nonzero(scored)[0], grid)
    distances.append(np.min(np.maximum(np.abs(yy[:, None] - ys[None]), np.abs(xx[:, None] - xs[None])), axis=1))
    targets = np.nan_to_num(spatial_targets[i], nan=0.0)
    leg_patches.append(np.nonzero(targets[:, family_idx].sum(axis=1) > 0)[0])  # keypoint patches whose target is a leg concept
    keypoint_patches.append(np.nonzero(scored & (targets[:, family_idx].sum(axis=1) == 0))[0])  # other part patches
distances = np.array(distances)
bands = [(0, "0, holds a keypoint"), (1, "1"), (2, "2"), (3, "3 or more")]
in_band = lambda d, b: (d >= 3) if b == 3 else (d == b)
pooled = [spatial_attention[:, :, c][in_band(distances, b)].mean() for b, _ in bands]
share_all = spatial_attention[:, :, c].mean() / spatial_attention.mean(axis=(0, 1)).sum()
won = int((spatial_attention.argmax(axis=2) == c).sum())
top_in = sum(int(spatial_attention[i].mean(axis=0).argmax() == c) for i in range(n_images))

with_legs = [i for i in range(n_images) if len(leg_patches[i])]
fig = plt.figure(figsize=(13.5, 1.9 * len(with_legs) + 0.9), constrained_layout=True)
outer = fig.add_gridspec(1, 2, width_ratios=[1.45, 1])
thumbs = outer[0].subgridspec(len(with_legs), N_THUMBS + 2)
for r, i in enumerate(with_legs):
    ax = fig.add_subplot(thumbs[r, 0])
    ax.imshow(images[i])
    for pch in leg_patches[i]:
        y, x = divmod(int(pch), grid)
        ax.add_patch(plt.Rectangle((x * PATCH, y * PATCH), PATCH, PATCH, fill=False, color=PALETTE["accent"], lw=1.3))
    ax.set_title("image {}".format(examples[i]["image_id"]), fontsize=9)
    ax.set_axis_off()
    for k, pch in enumerate(list(leg_patches[i])[:N_THUMBS]):
        y, x = divmod(int(pch), grid)
        ax = fig.add_subplot(thumbs[r, 1 + k])
        ax.imshow(images[i, y * PATCH:(y + 1) * PATCH, x * PATCH:(x + 1) * PATCH], interpolation="nearest")
        ax.set_title("leg keypoint patch\nslot: {:.0%}".format(spatial_attention[i, pch, c]), fontsize=8)
        ax.set_axis_off()
    pch = int(keypoint_patches[i][0])
    y, x = divmod(pch, grid)
    ax = fig.add_subplot(thumbs[r, N_THUMBS + 1])
    ax.imshow(images[i, y * PATCH:(y + 1) * PATCH, x * PATCH:(x + 1) * PATCH], interpolation="nearest")
    target_names = [spatial_attributes[k].split("::")[0].replace("has_", "").replace("_", " ")
                    for k in np.nonzero(np.nan_to_num(spatial_targets[i], nan=0.0)[pch])[0]]
    ax.set_title("{} patch\nfor comparison".format(target_names[0] if target_names else "other part"), fontsize=8)
    ax.set_axis_off()
ax = fig.add_subplot(outer[1])
x = np.arange(len(bands))
ax.bar(x, pooled, color=PALETTE["accent"], width=0.6, label="pooled over the six images")
for i in range(n_images):
    per_image = [spatial_attention[i, :, c][in_band(distances[i], b)].mean() if in_band(distances[i], b).any() else np.nan
                 for b, _ in bands]
    ax.plot(x, per_image, color="0.35", lw=0.9, marker="o", ms=3, alpha=0.6)
    if per_image[-1] > 0.02:  # the flat lines at zero are left unlabelled
        ax.text(x[-1] + 0.12, per_image[-1], str(examples[i]["image_id"]), fontsize=7.5, va="center", color="0.35")
ax.set_xticks(x)
ax.set_xticklabels([label for _, label in bands])
ax.set(xlabel="Patch distance from the nearest patch with a keypoint", ylabel="Attention on the slot", ylim=(0, 1),
       title="Attention on \u201c{}\u201d by distance from the bird".format(
           CONCEPT.replace("has_", "").replace("::", ": ").replace("_", " ")))
ax.legend(loc="upper left", fontsize=9)
ax.grid(False, axis="x")
plt.show()

caption("<b>Figure 6.2</b> Left: the patches at the leg keypoints (boxes) in the three images that have them, enlarged, beside "
        "a patch at another annotated part of the same bird. A leg crosses its patch as a line; the rest is ground. Right: "
        "attention on the slot by distance from the nearest patch with a keypoint, pooled over the six images (bars) and per "
        "image (lines): {:.0%} on keypoint patches, {:.0%} three or more patches away; the two lines at zero are the water "
        "images 3 and 2904, whose background goes to other slots. Cached tensors and images from the "
        "disjoint run's selected checkpoint.".format(pooled[0], pooled[-1]))
print("Over all {} patches of the six images, the slot takes {:.0%} of the spatial attention, is the largest weight in {} patches, "
      "and is the top concept over the whole image in {} of {}.".format(n_images * n_patches, share_all, won, top_in, n_images))
print("Label frequency in CUB: {} is present in {:.0%} of images, rank {} of {}; {} is present in {:.0%} and takes {:.1%} of the "
      "attention here.".format(CONCEPT, frequency[CONCEPT], int(np.nonzero(frequency.index == CONCEPT)[0][0]) + 1, n_spatial,
                                CONTRAST, frequency[CONTRAST],
                                spatial_attention[:, :, spatial_attributes.index(CONTRAST)].mean() / spatial_attention.mean(axis=(0, 1)).sum()))

### 6.3 Improvements and Next Steps

#### 6.3.1 Give every patch a target

Add a 96th spatial concept, "background", with its vote fixed at zero. With CUB's segmentation masks, every patch outside the bird targets it, and every patch inside the bird targets the concepts of the nearest part keypoint, so all 196 patches are scored. Background attention then appears as its own bar, with no vote, instead of under "leg color: black".

#### 6.3.2 Weight patches instead of averaging them

Replace `out_s.mean(1)` with a learned softmax weight per patch; the global stream already has one (`token_attention_pool`). Each patch's contribution becomes weight × attention × vote, still exact, and the weights show which patches the head counted.

#### 6.3.3 Penalize concentration only when the patches have a target

The concentration penalty is averaged over all 196 patches (`concepts_sparsity_cost` in `ctc/ctc.py`). Give it the keypoint mask the explanation loss already uses, so unscored patches are free to spread their attention.

#### 6.3.4 The labels are the handicap

The method's accuracy depends on the concept labels. The paper's Table 1 reports 88.0 ± 0.4 with them and 76.9 ± 3 without.

**What the labels are, on one species.** Every CUB image was labelled by one crowd worker who answered 312 yes/no questions about the bird and said how sure they were. The loader keeps 108 of the attributes and turns the answers into the training targets: an image's global target is the global attributes its worker marked yes, and its spatial targets are the part attributes marked yes, placed at the part's keypoint. Nothing else enters. The table shows what that gives for the Black-footed Albatross from Section 5: six single images, then the share of all 60 images labelled yes, then CUB's own class-level file for the species.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; instant: reads CUB's label files, no model &middot; change <code>SPECIES</code>, <code>ATTRIBUTES</code> or <code>SAMPLE_IMAGES</code></span></div>

In [ ]:
# One species' labels: six single images, the share of all its images labelled yes, and CUB's class-level file.
import pandas as pd
from data import CUB2011Parts_dataset

SPECIES = "Black_footed_Albatross"  # any species folder name from classes.txt, without its number
ATTRIBUTES = ["has_size::medium_(9_-_16_in)", "has_primary_color::brown", "has_shape::duck-like",  # global concepts
              "has_belly_pattern::solid", "has_leg_color::black"]  # spatial concepts
SAMPLE_IMAGES = [3, 1, 2, 4, 5, 6]  # image 3 is the Section 5 example; the others are the species' first images

CUB = DATA_DIR / "CUB_200_2011"
answers = pd.read_csv(CUB / "attributes" / "image_attribute_labels.txt", sep=r"\s+", header=None, engine="c",
                      names=["img", "attr", "present", "certainty", "time"], on_bad_lines="skip")
attribute_id = pd.read_csv(DATA_DIR / "attributes.txt", sep=" ", header=None, names=["attr", "name"]).set_index("name")["attr"]
species_id = pd.read_csv(CUB / "classes.txt", sep=" ", header=None, names=["cls", "name"]).set_index("name")["cls"]
image_species = pd.read_csv(CUB / "image_class_labels.txt", sep=" ", header=None, names=["img", "cls"]).set_index("img")["cls"]
class_file = np.loadtxt(CUB / "attributes" / "class_attribute_labels_continuous.txt")  # 200 species x 312 attributes, in percent
global_concepts = set(json.loads((ARTIFACT_DIR / "examples.json").read_text())["global_attributes"])
certainty = {1: "not visible", 2: "guessing", 3: "probably", 4: "definitely"}

cls = species_id[[name for name in species_id.index if name.endswith(SPECIES)][0]]
species_answers = answers[answers["img"].map(image_species) == cls]
assert all(image_species[img] == cls for img in SAMPLE_IMAGES), "SAMPLE_IMAGES must belong to SPECIES"


def answer(img, name):
    """One worker's answer for one image and attribute, with the certainty they gave."""
    row = species_answers[(species_answers["img"] == img) & (species_answers["attr"] == attribute_id[name])].iloc[0]
    if row["present"] == 1:
        return "yes, " + certainty[int(row["certainty"])]
    return "not visible" if row["certainty"] == 1 else "no"


def column(name):
    family, value = name.replace("has_", "").split("::")
    kind = "global" if name in global_concepts else "spatial"
    return "{}: {} ({})".format(family.replace("_", " "), value.split("_(")[0].replace("_", " "), kind)


rows = {}
for img in SAMPLE_IMAGES:
    rows["image {}{}".format(img, ", the Section 5 example" if img == SAMPLE_IMAGES[0] else "")] = [answer(img, n) for n in ATTRIBUTES]
n_species = species_answers["img"].nunique()
rows["all {} images: share labelled yes".format(n_species)] = [
    "{:.0%}".format(species_answers[species_answers["attr"] == attribute_id[n]]["present"].mean()) for n in ATTRIBUTES]
rows["CUB's class-level file"] = ["{:.0f}%".format(class_file[cls - 1, attribute_id[n] - 1]) for n in ATTRIBUTES]
table = pd.DataFrame.from_dict(rows, orient="index", columns=[column(n) for n in ATTRIBUTES])
table.index.name = SPECIES.replace("_", " ")
display(table)

# How sure the yes answers were, for this species and for all of CUB.
sure = species_answers[species_answers["present"] == 1]["certainty"].map(certainty).value_counts(normalize=True)
sure_all = answers[answers["present"] == 1]["certainty"].map(certainty).value_counts(normalize=True)
print("{}: of its {:,} yes answers, {:.0%} were given as 'definitely', {:.0%} 'probably', {:.0%} 'guessing'.".format(
    SPECIES.replace("_", " "), int((species_answers["present"] == 1).sum()),
    sure.get("definitely", 0), sure.get("probably", 0), sure.get("guessing", 0)))
print("All of CUB: {:,} yes answers; {:.0%} 'definitely', {:.0%} 'probably', {:.0%} 'guessing'.".format(
    int((answers["present"] == 1).sum()), sure_all.get("definitely", 0), sure_all.get("probably", 0), sure_all.get("guessing", 0)))

# How much the images of a species agree, over the 108 concepts the model learns.
kept = answers[answers["attr"].isin(CUB2011Parts_dataset.attributes_to_retain)].copy()
kept["cls"] = kept["img"].map(image_species)
rate = kept.groupby(["cls", "attr"])["present"].mean().unstack()  # share of each species' images labelled yes
per_image = pd.concat(  # of the concepts that characterise its species (yes in at least 45% of its images), the share an image carries
    group[group["attr"].isin(rate.columns[rate.loc[c] >= 0.45])].groupby("img")["present"].mean() for c, group in kept.groupby("cls"))
print("Over the 108 kept concepts, an image is labelled with {:.0%} of the concepts that characterise its species "
      "(median {:.0%}); the rest are labelled no or not visible.".format(per_image.mean(), per_image.median()))

Read down a column: the same species gets different targets from different workers. For "primary color: brown", the concept the model used most on image 3 in Section 5, images 1 and 2 are labelled no and images 3 to 5 yes, definitely. Image 3's worker also marked the bird not medium-sized and could not see its belly, while two thirds of the species' images are labelled medium and the class-level file gives a solid belly pattern 82%. The explanation loss trains toward each worker's answers as given, and across all of CUB 40% of the yes answers were given as "probably" or "guessing". That is the handicap: the model's ground truth for a concept is one person's quick answers per image, and most datasets have no such answers at all. The three points below reduce what the method needs from them.

##### 6.3.4.1 Get the targets without hand labels

Global concepts: CUB ships class-level vectors (`class_attribute_labels_continuous.txt`: for each class, the share of workers marking each attribute present), and the code ignores them. Replace each image's global target with its class's vector; the explanation loss normalizes its targets, so the shares work as they are. A new domain then needs one vector per class, written by an expert or a language model, instead of 312 decisions per image.

Spatial concepts: get part regions from an open-vocabulary segmenter prompted with the part names (beak, wing, leg, ...), and give every patch in a region the class's concepts for that part. The explanations then inherit the segmenter's errors instead of the workers' disagreement.

##### 6.3.4.2 Measure how much annotation the method needs

Train with 50%, 10% and 1% of the training images annotated, which needs only a loader change because the loss skips missing targets, and report accuracy and the on-bird attention share against the fraction. The paper says the gain is largest with little data and shows no such curve.

##### 6.3.4.3 Let the model find concepts, then name them

The code has a stream of concept slots without targets (`n_unsup_concepts`, default off for CUB; its forward pass routes through the supervised module). We can train it, then name each slot by the images and patches it fires on. This is still faithful, only that the names are given afterwards rather than before.